# Phase 4: Modelling

**CRISP-DM Phase Description:**  
In this phase, various modelling techniques are selected and applied, and their parameters are calibrated to optimal values. Typically, there are several techniques for the same data mining problem type, and some techniques have specific requirements on the form of the data. This may require stepping back to the Data Preparation phase.

---

In [1]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [3]:
# Load dataset
df = pd.read_csv("../data/raw/rainfall in india 1901-2015.csv")

# Verify dataset
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Dataset shape: 4116 rows x 19 columns


,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,332.6,388.5,558.2,33.6,3373.2,136.3,560.3,1696.3,980.3
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,666.2,197.2,359.0,160.5,3520.7,159.8,458.3,2185.9,716.7
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,339.0,181.2,284.4,225.0,2957.4,156.7,236.1,1874.0,690.6
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,820.4,222.2,308.7,40.1,3079.6,24.1,506.9,1977.6,571.0
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,297.0,260.7,25.4,344.7,2566.7,1.3,309.7,1624.9,630.8


# Task 1: Select Modelling Techniques

In [ ]:
modelling_techniques = {
    "problem_type": "Regression",
    "target_variable": "ANNUAL",
    "candidate_models": [
        {
            "name": "Linear Regression",
            "library": "sklearn.linear_model.LinearRegression",
            "justification": "Baseline regression model for rainfall prediction",
            "assumptions": "Assumes linear relationship; used for comparison"
        },
        {
            "name": "Random Forest Regressor",
            "library": "sklearn.ensemble.RandomForestRegressor",
            "justification": "Handles nonlinear rainfall patterns and interactions",
            "assumptions": "No strict assumptions; robust to outliers"
        },
        {
            "name": "Gradient Boosting Regressor",
            "library": "sklearn.ensemble.GradientBoostingRegressor",
            "justification": "Captures complex rainfall relationships",
            "assumptions": "Requires careful tuning"
        }
    ]
}

modelling_techniques

{'problem_type': 'Regression',
 'target_variable': 'ANNUAL',
 'candidate_models': [{'name': 'Linear Regression',
   'library': 'sklearn.linear_model.LinearRegression',
   'justification': 'Baseline regression model for rainfall prediction',
   'assumptions': 'Assumes linear relationship; used for comparison'},
  {'name': 'Random Forest Regressor',
   'library': 'sklearn.ensemble.RandomForestRegressor',
   'justification': 'Handles nonlinear rainfall patterns and interactions',
   'assumptions': 'No strict assumptions; robust to outliers'},
  {'name': 'Gradient Boosting Regressor',
   'library': 'sklearn.ensemble.GradientBoostingRegressor',
   'justification': 'Captures complex rainfall relationships',
   'assumptions': 'Requires careful tuning'}]}

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [8]:
# Remove non-numeric columns
df_numeric = df.select_dtypes(include=[np.number])

print("Numeric dataset shape:", df_numeric.shape)
df_numeric.head()

Numeric dataset shape: (4116, 18)


,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec
0,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,332.6,388.5,558.2,33.6,3373.2,136.3,560.3,1696.3,980.3
1,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,666.2,197.2,359.0,160.5,3520.7,159.8,458.3,2185.9,716.7
2,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,339.0,181.2,284.4,225.0,2957.4,156.7,236.1,1874.0,690.6
3,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,820.4,222.2,308.7,40.1,3079.6,24.1,506.9,1977.6,571.0
4,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,297.0,260.7,25.4,344.7,2566.7,1.3,309.7,1624.9,630.8


In [12]:
# Fill missing values
df_numeric = df.select_dtypes(include=[np.number])

df_numeric = df_numeric.fillna(df_numeric.median())

print("Remaining missing values:", df_numeric.isnull().sum().sum())

Remaining missing values: 0


# Task 2: Train/Test split

In [14]:
DATA_PATH = "../data/raw/rainfall in india 1901-2015.csv"
df = pd.read_csv(DATA_PATH)

TARGET_COL = "ANNUAL"

X = df_numeric.drop(columns=[TARGET_COL])
y = df_numeric[TARGET_COL]

RANDOM_SEED = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (3292, 17)
Test set: (824, 17)


# Task 3: Build Models

In [15]:
trained_models = {}

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
trained_models["Linear Regression"] = lr

# Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
rf.fit(X_train, y_train)
trained_models["Random Forest"] = rf

# Gradient Boosting
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
trained_models["Gradient Boosting"] = gb

print("Models trained successfully")

Models trained successfully


# Task 4: Assess Models

In [16]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).set_index("Model")
results_df

,MAE,RMSE,R2
Model,,,
Linear Regression,11.319930,49.310071,0.997222
Random Forest,43.063350,101.652590,0.988196
Gradient Boosting,46.671179,115.540340,0.984750


# Best Model

In [17]:
best_model_name = results_df["R2"].idxmax()
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

Best model: Linear Regression


# Cross Validation

In [18]:
cv_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("CV Scores:", cv_scores)
print("Mean R2:", cv_scores.mean())

CV Scores: [0.86351439 0.99962123 0.99310848 0.99986432 0.96718105]
Mean R2: 0.9646578917560433
